## ДЗ 3 Бойко Владимир МНАД 26/27

## Задание 1

In [1]:
%%bash
docker --version
docker compose version

Docker version 29.1.3, build 29.1.3-0ubuntu3~22.04.2
Docker Compose version 2.40.3+ds1-0ubuntu1~22.04.1


In [7]:
%%bash
cd ~/devops-hw/hw3/monitoring
sudo docker compose up -d 2>&1
sudo docker compose ps

 Container monitoring-prometheus-1  Running
 Container monitoring-node-exporter-1  Running
 Container monitoring-grafana-1  Running
NAME                         IMAGE                       COMMAND                  SERVICE         CREATED          STATUS          PORTS
monitoring-grafana-1         grafana/grafana:12.1.0      "/run.sh"                grafana         26 minutes ago   Up 26 minutes   127.0.0.1:3000->3000/tcp
monitoring-node-exporter-1   prom/node-exporter:v1.9.1   "/bin/node_exporter ��"   node-exporter   26 minutes ago   Up 26 minutes   
monitoring-prometheus-1      prom/prometheus:v3.5.0      "/bin/prometheus --c…"   prometheus      26 minutes ago   Up 21 minutes   127.0.0.1:9090->9090/tcp


In [3]:
%%bash
curl -s localhost:9100/metrics | grep '^node_' | head -n 15
echo "..."
echo "всего метрик node_*: $(curl -s localhost:9100/metrics | grep -c '^node_')"

node_arp_entries{device="docker0"} 1
node_arp_entries{device="eth0"} 1
node_boot_time_seconds 1.791926883e+09
node_context_switches_total 2.1371406e+07
node_cooling_device_cur_state{name="0",type="Processor"} 0
node_cooling_device_max_state{name="0",type="Processor"} 0
node_cpu_guest_seconds_total{cpu="0",mode="nice"} 0
node_cpu_guest_seconds_total{cpu="0",mode="user"} 0
node_cpu_seconds_total{cpu="0",mode="idle"} 315584.64
node_cpu_seconds_total{cpu="0",mode="iowait"} 2432.74
node_cpu_seconds_total{cpu="0",mode="irq"} 0
node_cpu_seconds_total{cpu="0",mode="nice"} 38.31
node_cpu_seconds_total{cpu="0",mode="softirq"} 8.05
node_cpu_seconds_total{cpu="0",mode="steal"} 54.99
node_cpu_seconds_total{cpu="0",mode="system"} 249.66
...
всего метрик node_*: 607


In [4]:
%%bash
sudo iptables -A INPUT -p tcp --dport 9100 -s 127.0.0.0/8 -j ACCEPT
sudo iptables -A INPUT -p tcp --dport 9100 -s 172.16.0.0/12 -j ACCEPT
sudo iptables -A INPUT -p tcp --dport 9100 -j DROP
sudo iptables -S INPUT | grep 9100

-A INPUT -s 127.0.0.0/8 -p tcp -m tcp --dport 9100 -j ACCEPT
-A INPUT -s 172.16.0.0/12 -p tcp -m tcp --dport 9100 -j ACCEPT
-A INPUT -p tcp -m tcp --dport 9100 -j DROP


In [5]:
%%bash
cd ~/devops-hw/hw3/monitoring
cat > prometheus/prometheus.yml <<'EOF'
global:
  scrape_interval: 15s        
  evaluation_interval: 15s    

scrape_configs:
  - job_name: prometheus            
    static_configs:
      - targets: [localhost:9090]

  - job_name: node                  
    static_configs:
      - targets: [host.docker.internal:9100]
        labels:
          instance: vm1             
EOF
cat prometheus/prometheus.yml

global:
  scrape_interval: 15s        
  evaluation_interval: 15s    

scrape_configs:
  - job_name: prometheus            
    static_configs:
      - targets: [localhost:9090]

  - job_name: node                  
    static_configs:
      - targets: [host.docker.internal:9100]
        labels:
          instance: vm1             


In [6]:
%%bash
cd ~/devops-hw/hw3/monitoring
sudo docker compose restart prometheus 2>&1
sleep 20   
echo "цели Prometheus"
curl -s localhost:9090/api/v1/targets | jq -r '.data.activeTargets[] | "\(.labels.job)\t\(.labels.instance)\t\(.health)\t\(.lastError)"'
echo "метрика up"
curl -s 'localhost:9090/api/v1/query?query=up' | jq -r '.data.result[] | "\(.metric.job)\t\(.metric.instance)\tup=\(.value[1])"'

 Container monitoring-prometheus-1  Restarting
 Container monitoring-prometheus-1  Started
цели Prometheus
node	vm1	up	
prometheus	localhost:9090	up	
метрика up
prometheus	localhost:9090	up=1
node	vm1	up=1


## Задание 2

In [11]:
%%bash
cd ~/devops-hw/hw3/counter-backend
ls -la
cat pyproject.toml | head -20

total 128
drwxrwxr-x 5 student student  4096 Oct 17 14:04 .
drwxrwxr-x 6 student student  4096 Oct 18 07:42 ..
-rw-rw-r-- 1 student student   171 Oct 17 14:04 .editorconfig
-rw-rw-r-- 1 student student   366 Oct 17 14:04 .gitignore
-rw-rw-r-- 1 student student   438 Oct 17 14:04 Makefile
-rw-rw-r-- 1 student student   692 Oct 17 14:04 README.md
drwxrwxr-x 2 student student  4096 Oct 17 14:04 deploy
-rw-rw-r-- 1 student student 87985 Oct 17 14:04 poetry.lock
-rw-rw-r-- 1 student student   646 Oct 17 14:04 pyproject.toml
drwxrwxr-x 3 student student  4096 Oct 17 14:04 src
drwxrwxr-x 2 student student  4096 Oct 17 14:04 tests
[tool.poetry]
name = "counter-backend"
version = "0.1.0"
description = ""
authors = []
readme = "README.md"
packages = [{ include = "counter_backend", from = "src" }]

[tool.poetry.scripts]
server = "counter_backend.main:main"

[tool.poetry.dependencies]
python = "^3.11"
aiohttp = "^3.8.5"
pytest-aiohttp = "^1.0.5"
pytest-asyncio = "^0.21.1"


[tool.poetry.group.dev.

In [12]:
%%bash
cd ~/devops-hw/hw3/counter-backend
cat > .dockerignore <<'EOF'

.git/
.gitignore
.editorconfig
Dockerfile*
.dockerignore

.venv/
__pycache__/
*.pyc
.pytest_cache/

dist/
build/
server.tar.gz
EOF
cat .dockerignore


.git/
.gitignore
.editorconfig
Dockerfile*
.dockerignore

.venv/
__pycache__/
*.pyc
.pytest_cache/

dist/
build/
server.tar.gz


In [13]:
%%bash
cd ~/devops-hw/hw3/counter-backend
cat > Dockerfile.single <<'EOF'
FROM python:3.11.6-slim-bookworm
SHELL ["/bin/sh", "-eu", "-c"]

WORKDIR /build
COPY pyproject.toml poetry.lock README.md ./
COPY src/ src/


RUN python -m pip install \
      --no-color \
      --no-cache-dir \
      --disable-pip-version-check \
      --no-python-version-warning \
      --no-warn-script-location \
      --break-system-packages \
      --progress-bar off \
      'poetry<2' setuptools wheel


RUN mkdir -p dist \
    && poetry export \
         --without-hashes \
         --format constraints.txt \
         --output dist/constraints.txt \
    && poetry run \
         python -m pip wheel \
           --isolated \
           --requirement dist/constraints.txt \
           --wheel-dir dist/vendor


RUN poetry build --format wheel

RUN packages=$(\
      find 'dist' 'dist/vendor' \
        -maxdepth 1 \
        -iname '*.whl' \
        -exec realpath {} \; \
        -print0 \
      | xargs --null) \
    && python -m pip install \
         --isolated \
         --no-index \
         --no-color \
         --no-cache-dir \
         --disable-pip-version-check \
         --no-python-version-warning \
         --no-warn-script-location \
         --no-deps \
         --break-system-packages \
         --progress-bar off \
         ${packages}

ENV PORT=8080
EXPOSE 8080/tcp
ENTRYPOINT ["python", "-m", "counter_backend"]
EOF
cat Dockerfile.single

FROM python:3.11.6-slim-bookworm
SHELL ["/bin/sh", "-eu", "-c"]

WORKDIR /build
COPY pyproject.toml poetry.lock README.md ./
COPY src/ src/


RUN python -m pip install \
      --no-color \
      --no-cache-dir \
      --disable-pip-version-check \
      --no-python-version-warning \
      --no-warn-script-location \
      --break-system-packages \
      --progress-bar off \
      'poetry<2' setuptools wheel


RUN mkdir -p dist \
    && poetry export \
         --without-hashes \
         --format constraints.txt \
         --output dist/constraints.txt \
    && poetry run \
         python -m pip wheel \
           --isolated \
           --requirement dist/constraints.txt \
           --wheel-dir dist/vendor


RUN poetry build --format wheel

RUN packages=$(\
      find 'dist' 'dist/vendor' \
        -maxdepth 1 \
        -iname '*.whl' \
        -exec realpath {} \; \
        -print0 \
      | xargs --null) \
    && python -m pip install \
         --isolated \
         --no-index \


In [14]:
%%bash
cd ~/devops-hw/hw3/counter-backend
sudo docker build -f Dockerfile.single -t counter-backend:single . 2>&1 | tail -n 30
sudo docker image ls counter-backend

  Attempting uninstall: packaging
    Found existing installation: packaging 26.3
    Uninstalling packaging-26.3:
      Successfully uninstalled packaging-26.3
  Attempting uninstall: idna
    Found existing installation: idna 3.20
    Uninstalling idna-3.20:
      Successfully uninstalled idna-3.20
  Attempting uninstall: charset-normalizer
    Found existing installation: charset-normalizer 3.5.2
    Uninstalling charset-normalizer-3.5.2:
      Successfully uninstalled charset-normalizer-3.5.2
 ---> Removed intermediate container a45baf802f33
 ---> 75ee4ff130c6
Step 10/12 : ENV PORT=8080
 ---> Running in 4a41febf8146
 ---> Removed intermediate container 4a41febf8146
 ---> 584c18dbfcc8
Step 11/12 : EXPOSE 8080/tcp
 ---> Running in 0437b3326f4b
 ---> Removed intermediate container 0437b3326f4b
 ---> 49333a2df136
Step 12/12 : ENTRYPOINT ["python", "-m", "counter_backend"]
 ---> Running in f7ed244ab0fd
 ---> Removed intermediate container f7ed244ab0fd
 ---> 0f2cad7d2553
Successfully bui

IMAGE                    ID             DISK USAGE   CONTENT SIZE   EXTRA
counter-backend:single   0f2cad7d2553        373MB         93.7MB        


In [16]:
%%bash
sudo docker rm -f cb-single >/dev/null 2>&1 || true
sudo docker run -d --name cb-single -p 127.0.0.1:8080:8080 counter-backend:single
sleep 3
echo "GET /api/counter"
curl -s -w '  (HTTP %{http_code})\n' localhost:8080/api/counter
echo "POST /api/counter"
curl -s -X POST -w '  (HTTP %{http_code})\n' localhost:8080/api/counter
echo "GET /api/counter"
curl -s -w '  (HTTP %{http_code})\n' localhost:8080/api/counter
echo "от какого пользователя работает процесс"
sudo docker exec cb-single id
echo "логи контейнера"
sudo docker logs cb-single 2>&1 | tail -n 5
sudo docker rm -f cb-single

78607c5f342d8dacaf17c5fb006ef19e0c7d1e27a85ad03a4b59dd68881763c7
GET /api/counter
{"count": 0}  (HTTP 200)
POST /api/counter
{"count": 1}  (HTTP 201)
GET /api/counter
{"count": 1}  (HTTP 200)
от какого пользователя работает процесс
uid=0(root) gid=0(root) groups=0(root)
логи контейнера
[INFO] 172.17.0.1 [18/Oct/2026:08:03:21 +0000] "GET /api/counter HTTP/1.1" 200 170 "-" "curl/7.81.0"
[INFO] 172.17.0.1 [18/Oct/2026:08:03:21 +0000] "POST /api/counter HTTP/1.1" 201 175 "-" "curl/7.81.0"
[INFO] 172.17.0.1 [18/Oct/2026:08:03:21 +0000] "GET /api/counter HTTP/1.1" 200 170 "-" "curl/7.81.0"
cb-single


In [17]:
%%bash
cd ~/devops-hw/hw3/counter-backend
cat > Dockerfile.multistage <<'EOF'

FROM python:3.11.6-slim-bookworm AS build
SHELL ["/bin/sh", "-eu", "-c"]

WORKDIR /build
COPY pyproject.toml poetry.lock README.md ./
COPY src/ src/

RUN python -m pip install \
      --no-color \
      --no-cache-dir \
      --disable-pip-version-check \
      --no-python-version-warning \
      --no-warn-script-location \
      --break-system-packages \
      --progress-bar off \
      'poetry<2' setuptools wheel


RUN mkdir -p dist \
    && poetry export \
         --without-hashes \
         --format constraints.txt \
         --output dist/constraints.txt \
    && poetry run \
         python -m pip wheel \
           --isolated \
           --requirement dist/constraints.txt \
           --wheel-dir dist/vendor

RUN poetry build --format wheel

FROM python:3.11.6-slim-bookworm AS final
SHELL ["/bin/sh", "-eu", "-c"]

WORKDIR /app
COPY --from=build /build/dist/ dist/

RUN packages=$(\
      find 'dist' 'dist/vendor' \
        -maxdepth 1 \
        -iname '*.whl' \
        -exec realpath {} \; \
        -print0 \
      | xargs --null) \
    && python -m pip install \
         --isolated \
         --no-index \
         --no-color \
         --no-cache-dir \
         --disable-pip-version-check \
         --no-python-version-warning \
         --no-warn-script-location \
         --no-deps \
         --break-system-packages \
         --progress-bar off \
         ${packages} \
    && rm -rf dist

ENV PORT=8080
EXPOSE 8080/tcp
ENTRYPOINT ["python", "-m", "counter_backend"]
EOF
cat Dockerfile.multistage


FROM python:3.11.6-slim-bookworm AS build
SHELL ["/bin/sh", "-eu", "-c"]

WORKDIR /build
COPY pyproject.toml poetry.lock README.md ./
COPY src/ src/

RUN python -m pip install \
      --no-color \
      --no-cache-dir \
      --disable-pip-version-check \
      --no-python-version-warning \
      --no-warn-script-location \
      --break-system-packages \
      --progress-bar off \
      'poetry<2' setuptools wheel


RUN mkdir -p dist \
    && poetry export \
         --without-hashes \
         --format constraints.txt \
         --output dist/constraints.txt \
    && poetry run \
         python -m pip wheel \
           --isolated \
           --requirement dist/constraints.txt \
           --wheel-dir dist/vendor

RUN poetry build --format wheel

FROM python:3.11.6-slim-bookworm AS final
SHELL ["/bin/sh", "-eu", "-c"]

WORKDIR /app
COPY --from=build /build/dist/ dist/

RUN packages=$(\
      find 'dist' 'dist/vendor' \
        -maxdepth 1 \
        -iname '*.whl' \
        -exec r

In [18]:
%%bash
cd ~/devops-hw/hw3/counter-backend
sudo docker build -f Dockerfile.multistage -t counter-backend:multistage . 2>&1 | tail -n 30
sudo docker image ls counter-backend

Processing ./dist/vendor/pytest_aiohttp-1.0.5-py3-none-any.whl
Processing ./dist/vendor/iniconfig-2.0.0-py3-none-any.whl
Processing ./dist/vendor/pluggy-1.3.0-py3-none-any.whl
Processing ./dist/vendor/aiosignal-1.3.1-py3-none-any.whl
Processing ./dist/vendor/idna-3.4-py3-none-any.whl
Processing ./dist/vendor/attrs-23.1.0-py3-none-any.whl
Processing ./dist/vendor/packaging-23.2-py3-none-any.whl
Processing ./dist/vendor/async_timeout-4.0.3-py3-none-any.whl
Processing ./dist/vendor/yarl-1.9.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing ./dist/vendor/pytest-7.4.3-py3-none-any.whl
Processing ./dist/vendor/charset_normalizer-3.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
 ---> Removed intermediate container fdd60a3c5ef3
 ---> 4011ce785d25
Step 14/16 : ENV PORT=8080
 ---> Running in 8974becc0ff8
 ---> Removed intermediate container 8974becc0ff8
 ---> b92c261f8ff2
Step 15/16 : EXPOSE 8080/tcp
 ---> Running in dc9ecaa8ab3d
 ---> Removed intermediate c

IMAGE                        ID             DISK USAGE   CONTENT SIZE   EXTRA
counter-backend:multistage   be4ce7d32cf9        234MB         58.8MB        
counter-backend:single       0f2cad7d2553        373MB         93.7MB        


In [20]:
%%bash
sudo docker rm -f cb-multistage >/dev/null 2>&1 || true
sudo docker run -d --name cb-multistage -p 127.0.0.1:8080:8080 counter-backend:multistage
sleep 3
echo "GET /api/counter"
curl -s -w '  (HTTP %{http_code})\n' localhost:8080/api/counter
echo "POST /api/counter"
curl -s -X POST -w '  (HTTP %{http_code})\n' localhost:8080/api/counter
echo "GET /api/counter"
curl -s -w '  (HTTP %{http_code})\n' localhost:8080/api/counter
echo "от какого пользователя работает процесс"
sudo docker exec cb-multistage id
echo "Poetry в образ не попал?"
sudo docker exec cb-multistage sh -c 'command -v poetry || echo "poetry нет — ок"'
sudo docker rm -f cb-multistage

f322ebef8b1fafc841b2da0425d87c5bee7010b3bcee2b1277f68d3421ba8757
GET /api/counter
{"count": 0}  (HTTP 200)
POST /api/counter
{"count": 1}  (HTTP 201)
GET /api/counter
{"count": 1}  (HTTP 200)
от какого пользователя работает процесс
uid=0(root) gid=0(root) groups=0(root)
Poetry в образ не попал?
poetry нет — ок
cb-multistage


In [21]:
%%bash
cd ~/devops-hw/hw3/counter-backend
cat > Dockerfile.nonroot <<'EOF'

FROM python:3.11.6-slim-bookworm AS build
SHELL ["/bin/sh", "-eu", "-c"]

WORKDIR /build
COPY pyproject.toml poetry.lock README.md ./
COPY src/ src/

RUN python -m pip install \
      --no-color \
      --no-cache-dir \
      --disable-pip-version-check \
      --no-python-version-warning \
      --no-warn-script-location \
      --break-system-packages \
      --progress-bar off \
      'poetry<2' setuptools wheel

RUN mkdir -p dist \
    && poetry export \
         --without-hashes \
         --format constraints.txt \
         --output dist/constraints.txt \
    && poetry run \
         python -m pip wheel \
           --isolated \
           --requirement dist/constraints.txt \
           --wheel-dir dist/vendor

RUN poetry build --format wheel

FROM python:3.11.6-slim-bookworm AS final
SHELL ["/bin/sh", "-eu", "-c"]

ARG USER=app
ARG HOME=/app
ARG UID=1001
ARG GID=1001
RUN \
  addgroup --gid "${GID}" "${USER}" \
  && adduser --disabled-password --gecos "" \
       --home "${HOME}" --ingroup "${USER}" --uid "${UID}" "${USER}"

WORKDIR ${HOME}
COPY --from=build /build/dist/ dist/

RUN packages=$(\
      find 'dist' 'dist/vendor' \
        -maxdepth 1 \
        -iname '*.whl' \
        -exec realpath {} \; \
        -print0 \
      | xargs --null) \
    && python -m pip install \
         --isolated \
         --no-index \
         --no-color \
         --no-cache-dir \
         --disable-pip-version-check \
         --no-python-version-warning \
         --no-warn-script-location \
         --no-deps \
         --break-system-packages \
         --progress-bar off \
         ${packages} \
    && rm -rf dist

USER ${USER}
ENV PORT=8080
EXPOSE 8080/tcp
ENTRYPOINT ["python", "-m", "counter_backend"]
EOF
cat Dockerfile.nonroot


FROM python:3.11.6-slim-bookworm AS build
SHELL ["/bin/sh", "-eu", "-c"]

WORKDIR /build
COPY pyproject.toml poetry.lock README.md ./
COPY src/ src/

RUN python -m pip install \
      --no-color \
      --no-cache-dir \
      --disable-pip-version-check \
      --no-python-version-warning \
      --no-warn-script-location \
      --break-system-packages \
      --progress-bar off \
      'poetry<2' setuptools wheel

RUN mkdir -p dist \
    && poetry export \
         --without-hashes \
         --format constraints.txt \
         --output dist/constraints.txt \
    && poetry run \
         python -m pip wheel \
           --isolated \
           --requirement dist/constraints.txt \
           --wheel-dir dist/vendor

RUN poetry build --format wheel

FROM python:3.11.6-slim-bookworm AS final
SHELL ["/bin/sh", "-eu", "-c"]

ARG USER=app
ARG HOME=/app
ARG UID=1001
ARG GID=1001
RUN \
  addgroup --gid "${GID}" "${USER}" \
  && adduser --disabled-password --gecos "" \
       --home "${HOME}

In [22]:
%%bash
cd ~/devops-hw/hw3/counter-backend
sudo docker build -f Dockerfile.nonroot -t counter-backend:nonroot . 2>&1 | tail -n 30
sudo docker image ls counter-backend

Processing ./dist/vendor/idna-3.4-py3-none-any.whl
Processing ./dist/vendor/attrs-23.1.0-py3-none-any.whl
Processing ./dist/vendor/packaging-23.2-py3-none-any.whl
Processing ./dist/vendor/async_timeout-4.0.3-py3-none-any.whl
Processing ./dist/vendor/yarl-1.9.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing ./dist/vendor/pytest-7.4.3-py3-none-any.whl
Processing ./dist/vendor/charset_normalizer-3.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
 ---> Removed intermediate container 7065da986b84
 ---> d668c53a1f93
Step 19/22 : USER ${USER}
 ---> Running in 7c66b4e27bda
 ---> Removed intermediate container 7c66b4e27bda
 ---> 3c2e2d97b158
Step 20/22 : ENV PORT=8080
 ---> Running in fc49ba6d540b
 ---> Removed intermediate container fc49ba6d540b
 ---> 5ce766f0a34f
Step 21/22 : EXPOSE 8080/tcp
 ---> Running in d995ccbb5acc
 ---> Removed intermediate container d995ccbb5acc
 ---> 4b0bcd86b2f7
Step 22/22 : ENTRYPOINT ["python", "-m", "counter_backend"]
 ---> Ru

IMAGE                        ID             DISK USAGE   CONTENT SIZE   EXTRA
counter-backend:multistage   be4ce7d32cf9        234MB         58.8MB        
counter-backend:nonroot      ec76a4f3d52e        234MB         58.8MB        
counter-backend:single       0f2cad7d2553        373MB         93.7MB        


In [23]:
%%bash
sudo docker rm -f cb-nonroot >/dev/null 2>&1 || true
sudo docker run -d --name cb-nonroot -p 127.0.0.1:8080:8080 counter-backend:nonroot
sleep 3
echo "GET /api/counter"
curl -s -w '  (HTTP %{http_code})\n' localhost:8080/api/counter
echo "POST /api/counter"
curl -s -X POST -w '  (HTTP %{http_code})\n' localhost:8080/api/counter
echo "GET /api/counter"
curl -s -w '  (HTTP %{http_code})\n' localhost:8080/api/counter
echo "от какого пользователя работает процесс"
sudo docker exec cb-nonroot id
sudo docker rm -f cb-nonroot

1f951461b14924076b544cdf3569ab8328389cc2d6dc6a98971d60bf2162cf17
GET /api/counter
{"count": 0}  (HTTP 200)
POST /api/counter
{"count": 1}  (HTTP 201)
GET /api/counter
{"count": 1}  (HTTP 200)
от какого пользователя работает процесс
uid=1001(app) gid=1001(app) groups=1001(app),100(users)
cb-nonroot


In [25]:
%%bash
echo "образы counter-backend"
sudo docker image ls counter-backend --format 'table {{.Tag}}\t{{.Size}}'
echo
echo "базовый образ"
sudo docker image ls python:3.11.6-slim-bookworm --format 'table {{.Repository}}:{{.Tag}}\t{{.Size}}'

образы counter-backend
TAG          SIZE
nonroot      234MB
multistage   234MB
single       373MB

базовый образ
REPOSITORY:TAG                SIZE
python:3.11.6-slim-bookworm   195MB


In [27]:
%%bash
for tag in single multistage nonroot; do
  echo "слои counter-backend:$tag"
  sudo docker history counter-backend:$tag --format '{{.Size}}\t{{.CreatedBy}}' | head -n 10 | cut -c1-110
  echo
done

слои counter-backend:single
0B	/bin/sh -eu -c #(nop)  ENTRYPOINT ["python" …
0B	/bin/sh -eu -c #(nop)  EXPOSE 8080/tcp
0B	/bin/sh -eu -c #(nop)  ENV PORT=8080
14.1MB	/bin/sh -eu -c packages=$(      find 'dist' …
16.4kB	/bin/sh -eu -c poetry build --format wheel
38MB	/bin/sh -eu -c mkdir -p dist     && poetry e…
80.7MB	/bin/sh -eu -c python -m pip install       -…
28.7kB	/bin/sh -eu -c #(nop) COPY dir:a211811e4e6f8…
106kB	/bin/sh -eu -c #(nop) COPY multi:0583ffee936…
8.19kB	/bin/sh -eu -c #(nop) WORKDIR /build

слои counter-backend:multistage
0B	/bin/sh -eu -c #(nop)  ENTRYPOINT ["python" …
0B	/bin/sh -eu -c #(nop)  EXPOSE 8080/tcp
0B	/bin/sh -eu -c #(nop)  ENV PORT=8080
26.5MB	/bin/sh -eu -c packages=$(      find 'dist' …
2.52MB	/bin/sh -eu -c #(nop) COPY dir:31435e171e0dd…
8.19kB	/bin/sh -eu -c #(nop) WORKDIR /app
0B	/bin/sh -eu -c #(nop)  SHELL [/bin/sh -eu -c]
0B	CMD ["python3"]
13.8MB	RUN /bin/sh -c set -eux;   savedAptMark="$(a…
0B	ENV PYTHON_GET_PIP_SHA256=9cc01665956d22b3bf…

сл